# 10 · Photon subtraction: optical Schrödinger kittens

Squeeze a mode, tap a small fraction of it on a beam splitter, and detect **one photon** in the
tap: the heralded state is non-Gaussian, an odd "kitten" close to a small cat state.

* Dakna, Anhut, Opatrný, Knöll, Welsch, *PRA* **55**, 3184 (1997): theory — the heralded state is
  exactly a squeezed single photon $S(r')|1\rangle$ with $\tanh r' = T\tanh r$.
* Ourjoumtsev, Tualle-Brouri, Laurat, Grangier, *Science* **312**, 83 (2006), and Neergaard-Nielsen
  *et al.*, *PRL* **97**, 083604 (2006): experiments, with negative Wigner functions limited by losses.

The pipeline is the hybrid one of `qttlind.circuits`: Gaussian backend (squeezer, beam splitter, detector
inefficiency) → Fock density matrix by the Bargmann recursion → photon-number projection →
loss → Wigner function. The dense oracle does the same with `expm` in a truncated Fock space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tutorial_utils as tu
from tutorial_utils import cd, dr

print(tu.cv_banner())
if tu.HAVE_GAUSSIAN:
    from qttlind.gaussian import GaussianState, ops, fock as gf, hafnian as hf
checks = {}
N0, N1 = 40, 6     # Fock cutoffs of the package route: output mode, tap mode
ND0, ND1 = 70, 8   # the oracle's own cutoffs: larger, then cropped, so that its truncation error is ~1e-10


def heralded_dense(r, theta, n_click=1, eta_tap=1.0):
    # oracle route: exact squeezed amplitudes, beam splitter by expm, optional Kraus loss on the tap,
    # then project the tap on |n_click>
    m = tu.DenseModes([ND0, ND1]).product_state(cd.squeezed_vacuum_amplitudes(ND0, r), cd.fock_ket(ND1, 0))
    m.gate("BS", (0, 1), theta)
    rho = np.outer(m.psi, m.psi.conj())
    if eta_tap < 1:
        rho = cd.loss_channel(rho, eta_tap, 1, [ND0, ND1])
    out = rho.reshape(ND0, ND1, ND0, ND1)[:, n_click, :, n_click]
    p = np.trace(out).real
    return out[:N0, :N0] / p, p


def heralded_gaussian(r, theta, eta_det=1.0, n_click=1):
    # package route: Gaussian prefix, Bargmann recursion, projection
    s = ops.beamsplitter(ops.squeeze(GaussianState.vacuum(2), r, 0.0, 0), theta)
    s = ops.loss(s, eta_det, mode=1)                      # detector inefficiency = loss before the detector
    rho = gf.fock_density_matrix(s, [N0, N1]).reshape(N0, N1, N0, N1)
    out = rho[:, n_click, :, n_click]
    p = np.trace(out).real
    return out / p, p

## 1. The heralded state is exactly a squeezed single photon

Because $a\,S(r)|0\rangle \propto S(r)|1\rangle$ and the beam splitter only rescales the squeezing seen by
the transmitted mode, the output is $S(r')|1\rangle$ with $\tanh r'=T\tanh r$, $T=\cos^2\theta$.

In [ ]:
r, theta = 0.6, 0.3
T = np.cos(theta) ** 2
rho_d, p_d = heralded_dense(r, theta)
rp = np.arctanh(T * np.tanh(r))
target = (cd.squeezing(200, rp) @ cd.fock_ket(200, 1))[:N0]
F = np.vdot(target, rho_d @ target).real
print(f"heralding probability {p_d:.6f};  fidelity with S(r')|1>, tanh r' = T tanh r: {F:.12f}")
checks["dense: heralded state = S(r')|1>"] = (abs(1 - F), 1e-9)
if tu.HAVE_GAUSSIAN:
    rho_g, p_g = heralded_gaussian(r, theta)
    s_tap = ops.beamsplitter(ops.squeeze(GaussianState.vacuum(2), r, 0.0, 0), theta).reduce([1])
    checks["Bargmann route vs dense route (state)"] = (np.abs(rho_g - rho_d).max(), 1e-8)
    checks["heralding probability: hafnian vs dense"] = (abs(hf.pattern_probability(s_tap, [1]) - p_d), 1e-9)

## 2. How good a kitten? Fidelity with an odd cat

$\max_{r'} |\langle \mathrm{cat}_-(\alpha)|S(r')|1\rangle|^2$ as a function of the cat amplitude, for a real $\alpha$:
the cat is elongated along $x$, so the best photon is squeezed along $p$ ($r'<0$ in the convention
$S(r)$ squeezes $x$). Above 0.99 up to $|\alpha|\approx 1$, then it drops — photon subtraction makes
*small* cats. (Photon subtraction from an $x$-squeezed vacuum, as in section 1, gives a cat along $p$.)

In [ ]:
alphas = np.linspace(0.2, 2.2, 21)
rs = np.linspace(-1.2, 1.2, 481)
Nf = 120
kets = [(cd.squeezing(Nf, x) @ cd.fock_ket(Nf, 1)) for x in rs]
best = []
for a in alphas:
    cat = dr.cat(Nf, a, -1)
    f = [abs(np.vdot(cat, k)) ** 2 for k in kets]
    best.append((max(f), rs[int(np.argmax(f))]))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(alphas, [b[0] for b in best], "o-")
ax.set_xlabel("|alpha| of the odd cat"); ax.set_ylabel("best fidelity"); ax.grid(alpha=0.3)
tu.savefig(fig, "10_kitten_fidelity.png")
i1 = int(np.argmin(abs(alphas - 1.0)))
print(f"alpha = 1: best r' = {best[i1][1]:+.3f}, fidelity {best[i1][0]:.4f}")
checks["kitten fidelity with the alpha = 1 odd cat > 0.99"] = (max(0.0, 0.99 - best[i1][0]), 0.0)

## 3. Wigner functions and the 50 % efficiency threshold

Loss $\eta$ maps parity to $(1-2\eta)^n$ (each photon survives with probability $\eta$), so for any state
with only odd photon numbers

$$
W(0) = \frac{2}{\pi}\sum_n p_n (1-2\eta)^n ,
$$

negative iff $\eta > 1/2$: an independent analytic check of the whole chain, and the reason why
the 2006 experiments needed overall efficiencies well above 50 %.

In [ ]:
grid = np.linspace(-2.5, 2.5, 51)
A = (grid[None, :] + 1j * grid[:, None]) / np.sqrt(2)          # alpha = (x + i p)/sqrt(2)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, eta in zip(axes, (1.0, 0.75, 0.5)):
    rho = cd.loss_channel(rho_d, eta, 0, [N0])
    W = cd.wigner_parity(rho, A) / 2                              # W(x, p)
    ax.contourf(grid, grid, W, 30, cmap="RdBu_r", vmin=-0.32, vmax=0.32)
    ax.set_title(f"eta = {eta}: W(0) = {W[25, 25]:+.3f}"); ax.set_aspect("equal"); ax.set_xlabel("x")
axes[0].set_ylabel("p"); tu.savefig(fig, "10_kitten_wigner.png")

etas = np.linspace(0.3, 1.0, 15)
pn = np.diag(rho_d).real
analytic = [(2 / np.pi) * np.sum(pn * (1 - 2 * e) ** np.arange(N0)) for e in etas]
numeric = [cd.wigner_parity(cd.loss_channel(rho_d, e, 0, [N0]), np.array([0.0]))[0] for e in etas]
checks["W(0) after loss: parity formula vs Kraus + Wigner"] = (np.abs(np.array(analytic) - numeric).max(), 1e-12)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(etas, numeric, "o", label="Kraus loss + parity Wigner (oracle)")
ax.plot(etas, analytic, "-", label="(2/pi) sum p_n (1-2 eta)^n")
ax.axvline(0.5, color="k", lw=0.5); ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("efficiency eta after heralding"); ax.set_ylabel("W(0) (alpha units)"); ax.legend()
tu.savefig(fig, "10_w0_threshold.png");

## 4. A digital-twin knob: detector inefficiency

An inefficient photon counter is an ideal one behind a loss $\eta_d$. With $\eta_d<1$, two-photon events
can be recorded as one: the heralded state becomes a mixture, and the negativity fills in even
without any loss on the output. (Needs the Gaussian backend: the inefficiency is a Gaussian
channel inserted *before* the non-Gaussian projection.)

In [ ]:
if tu.HAVE_GAUSSIAN:
    eds = np.linspace(0.1, 1.0, 10)
    w0, ph = [], []
    for ed in eds:
        rho_e, p_e = heralded_gaussian(0.6, 0.3, eta_det=ed)
        w0.append(cd.wigner_parity(rho_e, np.array([0.0]))[0]); ph.append(p_e)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(eds, w0, "o-", label="W(0)"); ax2 = ax.twinx(); ax2.plot(eds, ph, "s--", color="C1", label="P(herald)")
    ax.set_xlabel("detector efficiency"); ax.set_ylabel("W(0)"); ax2.set_ylabel("heralding probability")
    tu.savefig(fig, "10_detector_efficiency.png")
    # cross-check one point against the dense oracle: Kraus loss on the tap before projecting
    rho2, _ = heralded_dense(0.6, 0.3, eta_tap=0.5)
    checks["inefficient detector: Gaussian route vs dense"] = (np.abs(heralded_gaussian(0.6, 0.3, 0.5)[0] - rho2).max(), 1e-8)
else:
    print("qttlind.gaussian is not available: detector-inefficiency scan skipped.")

## 5. The same experiment as a circuit

The `qttlind.circuits` executors run the experiment from its description. The hybrid executor
keeps the squeezer and the beam splitter on covariance matrices and moves the two (correlated)
modes to the comb only at the photon detection; its log records the hand-off. The all-comb
`QTTExecutor` must agree, and a hardware model adds detector inefficiency.

In [ ]:
if tu.HAVE_CIRCUITS:
    from qttlind.circuits import Circuit, CVMode, HardwareModel, HybridExecutor, QTTExecutor
    circ = Circuit(2).squeeze(0, 0.6).beamsplitter(0, 1, 0.3).measure_fock(1, postselect=1)
    cv_modes = [CVMode.fock("out", 6), CVMode.fock("tap", 4)]
    hyb = HybridExecutor(cv_modes).run(circ)
    qtt = QTTExecutor(cv_modes).run(circ)
    print("\n".join(hyb.backend_log))
    rho_h = hyb.state.reduced_dm(0)[:N0, :N0]
    _, p_g = heralded_gaussian(0.6, 0.3)
    checks["circuit (hybrid): heralding probability"] = (abs(hyb.probability - p_g), 1e-10)
    checks["circuit (QTT) vs hybrid: probability"] = (abs(qtt.probability - hyb.probability), 1e-8)
    w0 = 2 / np.pi * hyb.state.expect({0: cv_modes[0].parity()}).real
    checks["circuit: W(0) = -2/pi"] = (abs(w0 + 2 / np.pi), 1e-8)
    twin = HardwareModel(detector_efficiency=0.7).compile(circ)
    res_tw = HybridExecutor(cv_modes).run(twin)
    rho_e, p_e = heralded_gaussian(0.6, 0.3, eta_det=0.7)
    checks["digital twin: probability with eta_d = 0.7"] = (abs(res_tw.probability - p_e), 1e-10)
    print(f"p_herald = {hyb.probability:.10f} (ideal), {res_tw.probability:.10f} (eta_d = 0.7); W(0) = {w0:.6f}")

## Acceptance table

In [ ]:
for name, (dev, tol) in checks.items():
    print(f"{'ok ' if dev <= tol else 'FAIL'} {name:<52s} deviation {dev:9.2e}  (tolerance {tol:.0e})")
assert all(dev <= tol for dev, tol in checks.values()), "a benchmark is outside its tolerance"